<a href="https://colab.research.google.com/github/Kei300/P2-IA-Python-Libraries-Statistics/blob/main/2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Contaminantes CDMX Partículas PM2.5

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
df = pd.read_csv('/content/contaminantes_2026.csv', header=9, on_bad_lines='skip')

Tabla con renglón por estación-hora y una columna por contaminante

In [3]:
df_contaminantes = df.pivot(
    index=['date', 'id_station'],
    columns='id_parameter',
    values='valor'
)
df_contaminantes.columns = df_contaminantes.columns.astype(str)
df_contaminantes = df_contaminantes.dropna(how='all')
display(df_contaminantes)

id_parameter                      CO   NO   NO2   NOX    O3  PM10  PM2.5  \
date                id_station                                             
2026-01-01 00:00:00 AJM         0.74  2.0  26.0  28.0  19.0  67.0   53.0   
                    ATI         0.70  2.0  21.0  23.0  18.0  90.0    NaN   
                    BJU         1.07  NaN  32.0   NaN  17.0  62.0   41.0   
                    CAM          NaN  7.0  44.0  52.0   4.0  57.0   49.0   
                    CCA         1.12  1.0  32.0  33.0  13.0   NaN   27.0   
...                              ...  ...   ...   ...   ...   ...    ...   
2026-07-31 23:00:00 SAC         0.31  2.0  17.0  18.0  34.0   NaN    8.0   
                    TAH         0.54  1.0  15.0  16.0  28.0   NaN    NaN   
                    TLA         0.57  3.0  40.0  43.0  17.0  27.0   22.0   
                    TLI          NaN  0.0  24.0  24.0   NaN   3.0    NaN   
                    UIZ         0.45  0.0  24.0  24.0  24.0  16.0   13.0   

id_parameter                    PMCO   SO2  
date                id_station              
2026-01-01 00:00:00 AJM         14.0  28.0  
                    ATI          NaN  22.0  
                    BJU         21.0   3.0  
                    CAM          8.0  25.0  
                    CCA          NaN   3.0  
...                              ...   ...  
2026-07-31 23:00:00 SAC          NaN   7.0  
                    TAH          NaN   4.0  
                    TLA          4.0   2.0  
                    TLI          NaN   2.0  
                    UIZ          3.0   5.0  

[133822 rows x 9 columns]

Covarianza y correlación de CO con NOX ¿Son distintas?

In [4]:
covarianza = df_contaminantes['CO'].cov(df_contaminantes['NOX'])
correlacion= df_contaminantes['CO'].corr(df_contaminantes['NOX'])

print(f"Covarianza CO vs NOX: {covarianza:.4f}")
print(f"Correlación CO vs NOX: {correlacion:.4f}")

# Son distintas. La covarianza depende de las unidades de medida de las variables,
# mientras que la correlación es la covarianza estandarizada
# en el rango [-1, 1], lo que ayuda a su análisis.

Covarianza CO vs NOX: 11.5867
Correlación CO vs NOX: 0.8656


Eigenvalores de la matriz de correlación, con y sin faltantes

In [5]:
df_completos = df_contaminantes.dropna()
corr_sin_faltantes = df_completos.corr()
eigvals_sin_faltantes = np.linalg.eigvals(corr_sin_faltantes)

In [6]:
df_imputado = df_contaminantes.fillna(df_contaminantes.mean())
corr_con_faltantes = df_imputado.corr()
eigvals_con_faltantes = np.linalg.eigvals(corr_con_faltantes)

In [7]:
print("=== Eigenvalores ===")
print(f"{'Comp.':<6} | {'Sin Faltantes':<14} | {'Con Faltantes':<14}")
print("-" * 40)

for i, (sin, con) in enumerate(zip(np.sort(eigvals_sin_faltantes)[::-1], np.sort(eigvals_con_faltantes)[::-1]), 1):
    print(f"PC{i:<4} | {sin:<14.4f} | {con:<14.4f}")

=== Eigenvalores ===
Comp.  | Sin Faltantes  | Con Faltantes 
----------------------------------------
PC1    | 4.4557         | 3.5628        
PC2    | 1.9783         | 1.7435        
PC3    | 0.9122         | 0.9543        
PC4    | 0.6445         | 0.7963        
PC5    | 0.5526         | 0.6433        
PC6    | 0.3342         | 0.5206        
PC7    | 0.1220         | 0.4825        
PC8    | 0.0004         | 0.2859        
PC9    | 0.0001         | 0.0106        


PM2.5 a partir de los demás contaminantes, luego solo contra gases. Cakcykar $R^2$ y VIF

In [8]:
X_todos = df_completos.drop(columns=['PM2.5'])
y_pm25 = df_completos['PM2.5']

X_todos_const = sm.add_constant(X_todos)
modelo_todos = sm.OLS(y_pm25, X_todos_const).fit()

print(f"R² (Todos los contaminantes): {modelo_todos.rsquared:.4f}")

vif_todos = {col: variance_inflation_factor(X_todos_const.values, i)
             for i, col in enumerate(X_todos_const.columns) if col != 'const'}

df_vif_todos = pd.DataFrame({
    'Contaminante': list(vif_todos.keys()),
    'VIF': [float(val) for val in vif_todos.values()]
})
df_vif_todos['VIF'] = df_vif_todos['VIF'].map('{:.2f}'.format)

display(df_vif_todos)

R² (Todos los contaminantes): 0.9983


,Contaminante,VIF
0,CO,5.71
1,NO,2324.24
2,NO2,780.73
3,NOX,4528.15
4,O3,1.77
5,PM10,4.54
6,PMCO,3.40
7,SO2,1.11


In [9]:
gases = ['CO', 'NO', 'NO2', 'NOX', 'O3', 'SO2']
X_gases = df_completos[gases]

X_gases_const = sm.add_constant(X_gases)
modelo_gases = sm.OLS(y_pm25, X_gases_const).fit()

vif_gases = {col: variance_inflation_factor(X_gases_const.values, i)
             for i, col in enumerate(X_gases_const.columns) if col != 'const'}

print(f"\nR² (Solo gases): {modelo_gases.rsquared:.4f}")

df_vif_gases = pd.DataFrame({
    'Contaminante': list(vif_gases.keys()),
    'VIF': [float(val) for val in vif_gases.values()]
})
df_vif_gases['VIF'] = df_vif_gases['VIF'].map('{:.2f}'.format)

display(df_vif_gases)


R² (Solo gases): 0.4055


,Contaminante,VIF
0,CO,5.58
1,NO,2324.15
2,NO2,780.53
3,NOX,4527.80
4,O3,1.34
5,SO2,1.04


H con una estación y una semana: comprueben $H^2$ = H y traza = p. Luego el leverage de todos sin construir H

In [10]:
df_uiz_semana1 = df_contaminantes.xs('UIZ', level='id_station').dropna().loc['2026-01-01':'2026-01-07']

X_semana1_gases = sm.add_constant(df_uiz_semana1[gases]).values

# H = X (Xᵀ X)⁻¹ Xᵀ
matriz_H = X_semana1_gases @ np.linalg.inv(X_semana1_gases.T @ X_semana1_gases) @ X_semana1_gases.T

print(f"¿Se cumple H² = H?: {np.allclose(matriz_H @ matriz_H, matriz_H)}")
print(f"Traza de H: {np.trace(matriz_H):.4f} | Parámetros p: {X_semana1_gases.shape[1]}")

¿Se cumple H² = H?: True
Traza de H: 7.0000 | Parámetros p: 7


In [11]:
X_global = sm.add_constant(df_completos[gases]).values
X_inv_global = np.linalg.inv(X_global.T @ X_global)
leverage_valores = np.sum((X_global @ X_inv_global) * X_global, axis=1)

df_completos = df_contaminantes.dropna().copy()
df_completos['leverage'] = leverage_valores

display(df_completos[['leverage']].describe())

id_parameter,leverage
count,17955.000000
mean,0.000390
std,0.000598
min,0.000059
25%,0.000163
50%,0.000256
75%,0.000417
max,0.022124


Cook: ¿los 10 más influyentes son los de más leverage? Reajusten sin ellos

In [12]:
influencia = modelo_gases.get_influence()
df_completos['distancia_cook'] = influencia.cooks_distance[0]

top10_cook = df_completos.nlargest(10, 'distancia_cook').index
top10_leverage = df_completos.nlargest(10, 'leverage').index

coincidencias = set(top10_cook).intersection(set(top10_leverage))

df_filtrado_sin_outliers = df_completos.drop(index=top10_cook)

X_reajuste = sm.add_constant(df_filtrado_sin_outliers[gases])
y_reajuste = df_filtrado_sin_outliers['PM2.5']

modelo_reajustado = sm.OLS(y_reajuste, X_reajuste).fit()

print(f"Coincidencias entre Top 10 Cook y Top 10 Leverage: {len(coincidencias)}/10")
print(f"\n Comparación de R² al reajustar el modelo de gases:")
print(f"   - R² original (con outliers): {modelo_gases.rsquared:.6f}")
print(f"   - R² reajustado (sin top 10 Cook): {modelo_reajustado.rsquared:.6f}")

Coincidencias entre Top 10 Cook y Top 10 Leverage: 4/10

 Comparación de R² al reajustar el modelo de gases:
   - R² original (con outliers): 0.405506
   - R² reajustado (sin top 10 Cook): 0.430860


In [13]:
# Tener un alto leverage solo significa que el valor de las X es atípico, pero no
# necesariamente que altere el modelo. La Distancia de Cook evalúa tanto el leverage
# como el tamaño del residuo para medir el impacto real en los coeficientes.
#
# Solo 4 de los 10 puntos con más leverage resultaron ser influyentes. Al eliminar
# los 10 puntos con mayor Distancia de Cook, el R² del modelo mejora de 0.4055 a 0.4309,
# confirmando quitar los outliers es beneficioso.